### PHASE 2 — Tuesday | Geographic Risk Analysis

---

> `Business Request from CEO:`
> "Where are we bleeding money? Which cities have the worst default rates? I need to know which geographic markets are causing this crisis so we can adjust our regional strategies."

> `Tuesday Mindset Unlock:` 
> Yesterday you learned the scale. Today you find where. The most dangerous mistake here is sorting by default rate alone. A city with 25% defaults on 20 loans is not the same problem as a city with 12% defaults on 2,000 loans. Rate tells you frequency. Exposure tells you cost. Only the product of both tells you where to act first. If you sort by rate — you get the wrong halt list. An analyst sorts by impact.

> `CEO'S GEOGRAPHIC AUDIT — "Which cities are destroying our portfolio?"`
> We need to know where defaults are concentrated geographically. The CEO wants a top city halt recommendation — but the decision requires more than a ranked table. A business can't simply "stop lending in Mumbai" without understanding the exposure, the active loan pipeline, and the statistical validity of the signal.

---

Dataset Identification

- Where are we bleeding money geographically?
- Which cities have the worst default rates?
- Which geographic markets are causing the crisis?

> Identified datasets: *loans.csv*, *customers.csv*, *dim_city.csv*, *dim_state.csv*

---

### Start Spark Session

In [1]:
## Import dependencies and create Spark session
import time
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/25 10:13:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Global Variables

In [2]:
N = 20 # Number of rows to show in results

### Load the data

In [3]:
## Load the identified datasets and create a temporary views for SQL queries
loans = spark.read.csv("../datasets/loans.csv", header=True)
customers = spark.read.csv("../datasets/customers.csv", header=True)
dim_city = spark.read.csv("../datasets/dim_city.csv", header=True)
dim_state = spark.read.csv("../datasets/dim_state.csv", header=True)

loans.createOrReplaceTempView("loans")
customers.createOrReplaceTempView("customers")
dim_city.createOrReplaceTempView("dim_city")
dim_state.createOrReplaceTempView("dim_state")

print("Loans dataset:")
loans.show(n=2)
print("Customers dataset:")
customers.show(n=2)
print("Dimensional City dataset:")
dim_city.show(n=2)
print("Dimensional State dataset:")
dim_state.show(n=2)

Loans dataset:
+-------+-----------+--------------+-----------+-----------+-------------+------------------+----------------+-----------------+-------------+-----------+--------------------+
|loan_id|customer_id|institution_id|loan_amount|loan_status|interest_rate|loan_tenure_months|application_date|disbursement_date|maturity_date| emi_amount|     purpose_of_loan|
+-------+-----------+--------------+-----------+-----------+-------------+------------------+----------------+-----------------+-------------+-----------+--------------------+
|      1|       2440|          2954| 190607.125|  Defaulted|  11.39000034|                84|      08-12-2021|       23-12-2021|   16-11-2028|3302.540039|     Living Expenses|
|      2|       2440|          4741| 425798.375|     Active|  14.43999958|                48|      01-01-2022|       11-01-2022|   21-12-2025|11728.73047|Course Fees + Living|
+-------+-----------+--------------+-----------+-----------+-------------+------------------+------------

---

STEP 2A — City-Level Loan Volume

What you're doing: Count total loans per city. This tells you where EduFin is most active geographically. You need this before defaults — because volume determines whether a default signal is meaningful or noise.

*Hint: JOIN loans to dim_city, GROUP BY city_name. COUNT loans per city, ORDER BY total_loans DESC. Decide: INNER JOIN or LEFT JOIN* 

In [4]:
query = """
SELECT 
    dc.city_name AS `City`,
    COUNT(*) AS `Loans`
FROM loans l
INNER JOIN customers c ON l.customer_id = c.customer_id
INNER JOIN dim_city dc ON c.city_id = dc.city_id
GROUP BY dc.city_name
ORDER BY `Loans` DESC
"""
start = time.perf_counter()
spark.sql(query).show(n=N)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+-----------+-----+
|       City|Loans|
+-----------+-----+
|      Delhi|  209|
|     Raipur|  207|
|     Bhopal|  204|
|Bhubaneswar|  199|
|    Chennai|  191|
|      Patna|  183|
|    Lucknow|  179|
|     Nagpur|  176|
|   Guwahati|  176|
|     Indore|  170|
|     Ranchi|  168|
|     Mumbai|  168|
| Chandigarh|  168|
|     Rajkot|  166|
|      Kochi|  165|
|   Dehradun|  163|
|  Ahmedabad|  160|
|     Jaipur|  160|
|      Jammu|  159|
|     Kanpur|  159|
+-----------+-----+
only showing top 20 rows
Time: 1.271 seconds


How many cities have fewer than 50 loans? These are candidates for the "insufficient data" exclude list later.


In [5]:
query = """
SELECT 
    dc.city_name AS `City`,
    COUNT(*) AS `Loans`
FROM loans l
INNER JOIN customers c ON l.customer_id = c.customer_id
INNER JOIN dim_city dc ON c.city_id = dc.city_id
GROUP BY dc.city_name
HAVING COUNT(*) < 100
"""
start = time.perf_counter()
spark.sql(query).show(n=N)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+----+-----+
|City|Loans|
+----+-----+
+----+-----+

Time: 0.558 seconds


Query 2A: Geographic Borrower Distribution

- Calculate: Number of customers and loans per city.
- Business Purpose: Identify our largest markets by volume.
- Filter: Focus on disbursed loans only.

In [6]:
query = """
WITH city_loans AS (
    SELECT 
        dc.city_name AS city,
        COUNT(*) AS loans,
        COUNT(DISTINCT c.customer_id) AS customers,
        SUM(l.loan_amount) AS total_amount
    FROM loans l
    INNER JOIN customers c ON l.customer_id = c.customer_id
    INNER JOIN dim_city dc ON c.city_id = dc.city_id
    GROUP BY dc.city_name
)
SELECT
    city AS `City`,
    loans AS `Loans`,
    customers AS `Customers`,
    CONCAT(ROUND(total_amount * 100.0 / SUM(total_amount) OVER (), 2), '%') AS `Portfolio Share (%)`
FROM city_loans
ORDER BY loans DESC;
"""
start = time.perf_counter()
spark.sql(query).show(n=N)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+-----------+-----+---------+-------------------+
|       City|Loans|Customers|Portfolio Share (%)|
+-----------+-----+---------+-------------------+
|      Delhi|  209|      115|              4.01%|
|     Raipur|  207|      119|              4.15%|
|     Bhopal|  204|      118|              4.16%|
|Bhubaneswar|  199|      116|              4.13%|
|    Chennai|  191|      119|              3.73%|
|      Patna|  183|      114|              3.79%|
|    Lucknow|  179|      101|              3.67%|
|     Nagpur|  176|      110|              3.44%|
|   Guwahati|  176|       99|              3.56%|
|     Indore|  170|      104|              3.29%|
|     Ranchi|  168|      103|              3.27%|
|     Mumbai|  168|      106|              3.22%|
| Chandigarh|  168|       97|              3.46%|
|     Rajkot|  166|      106|              3.09%|
|      Kochi|  165|      105|              3.26%|
|   Dehradun|  163|       97|              3.16%|
|  Ahmedabad|  160|       97|              3.45%|


---

STEP 2B — Default Count and Rate Per City

What you're doing: Calculate total defaults and default rate per city. This is the first time a rate and a volume are in the same row — which means the relationship between them becomes visible.

*Hint: COUNT total, COUNT WHERE status=Defaulted, ROUND(defaulted/total * 100, 2) AS default_rate. GROUP BY city_name.*

In [7]:
query = """
SELECT 
    dc.city_name AS `City`,
    ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END), 2) AS `Defaulted Loans`,
    CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) / COUNT(*) * 100, 2), '%') AS `Default Rates (%)`
FROM loans l
INNER JOIN customers c ON l.customer_id = c.customer_id
INNER JOIN dim_city dc ON c.city_id = dc.city_id
GROUP BY dc.city_name;
"""
start = time.perf_counter()
spark.sql(query).show(n=N)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+-----------+---------------+-----------------+
|       City|Defaulted Loans|Default Rates (%)|
+-----------+---------------+-----------------+
|  Bangalore|             13|            8.78%|
|      Kochi|             25|           15.15%|
|Bhubaneswar|             19|            9.55%|
|      Jammu|             23|           14.47%|
|      Patna|             17|            9.29%|
|   Vadodara|             11|            7.19%|
|    Gangtok|             17|           11.33%|
|    Chennai|             13|            6.81%|
|    Lucknow|             25|           13.97%|
|     Shimla|             19|           13.19%|
|     Ranchi|             30|           17.86%|
|     Mumbai|             22|            13.1%|
|  Ahmedabad|             24|            15.0%|
|    Kolkata|             11|            8.53%|
|   Dehradun|             16|            9.82%|
|       Pune|             30|           20.13%|
|      Delhi|             22|           10.53%|
|     Raipur|             21|           

Which city has the highest default RATE? Which has the highest total defaults? Are they the same city? If not — which matters more?

> Highest default rate

In [8]:
query = """
SELECT 
    'Highest Default Rate' AS `Metric`,
    dc.city_name AS `City`,
    SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) AS `Defaulted Loan`,
    ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS `Default Rate (%)`
FROM loans l
INNER JOIN customers c ON l.customer_id = c.customer_id
INNER JOIN dim_city dc ON c.city_id = dc.city_id
GROUP BY dc.city_name
ORDER BY SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100.0 / COUNT(*) DESC
LIMIT 1
"""
start = time.perf_counter()
spark.sql(query).show(n=N)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+--------------------+----+--------------+----------------+
|              Metric|City|Defaulted Loan|Default Rate (%)|
+--------------------+----+--------------+----------------+
|Highest Default Rate|Pune|            30|           20.13|
+--------------------+----+--------------+----------------+

Time: 0.468 seconds


> Highest total defaults

In [9]:
query = """
SELECT 
    'Highest Total Defaults' AS `Metric`,
    dc.city_name AS `City`,
    SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) AS `Defaulted Loan`,
    ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS `Default Rate (%)`
FROM loans l
INNER JOIN customers c ON l.customer_id = c.customer_id
INNER JOIN dim_city dc ON c.city_id = dc.city_id
GROUP BY dc.city_name
ORDER BY SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) DESC
LIMIT 1
"""
start = time.perf_counter()
spark.sql(query).show(n=N)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+--------------------+------+--------------+----------------+
|              Metric|  City|Defaulted Loan|Default Rate (%)|
+--------------------+------+--------------+----------------+
|Highest Total Def...|Ranchi|            30|           17.86|
+--------------------+------+--------------+----------------+

Time: 0.355 seconds


- Ranchi: Highest total defaults (30 loans, ₹1.40 Cr) — absolute damage
- Pune: Highest default rate (20.13%) — quality signal
- Why both matter: Rate shows frequency, amount shows cost. Only impact (rate × amount) tells you where to act first.

Query 2B: City-Level Risk Metrics

- Calculate: Default rate per city.
- Business Purpose: Pinpoint which specific locations are underperforming.
- Filter: Exclude cities with low volume (<100 loans) to avoid statistical noise.

In [10]:
query = """
SELECT 
    dc.city_name AS `City`,
    COUNT(*) AS `Loans`,
    SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) AS `Defaults`,
    CONCAT(
        ROUND(
            (SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100.0 / COUNT(*)), 
            2
        ), 
        '%'
    ) AS `Default Rates (%)`
FROM loans l
INNER JOIN customers c ON l.customer_id = c.customer_id
INNER JOIN dim_city dc ON c.city_id = dc.city_id
GROUP BY dc.city_name
HAVING COUNT(*) >= 100
ORDER BY SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100.0 / COUNT(*) DESC;
"""
start = time.perf_counter()
spark.sql(query).show(n=N)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+----------+-----+--------+-----------------+
|      City|Loans|Defaults|Default Rates (%)|
+----------+-----+--------+-----------------+
|      Pune|  149|      30|           20.13%|
|    Ranchi|  168|      30|           17.86%|
|    Indore|  170|      28|           16.47%|
|    Jaipur|  160|      26|           16.25%|
|    Rajkot|  166|      26|           15.66%|
|     Kochi|  165|      25|           15.15%|
| Ahmedabad|  160|      24|           15.00%|
|     Jammu|  159|      23|           14.47%|
|    Nashik|  135|      19|           14.07%|
|   Lucknow|  179|      25|           13.97%|
|    Shimla|  144|      19|           13.19%|
|    Mumbai|  168|      22|           13.10%|
|    Nagpur|  176|      22|           12.50%|
|   Gangtok|  150|      17|           11.33%|
|    Bhopal|  204|      22|           10.78%|
|Chandigarh|  168|      18|           10.71%|
|     Delhi|  209|      22|           10.53%|
|    Raipur|  207|      21|           10.14%|
| Hyderabad|  159|      16|       

---

STEP 2C — Financial Exposure Per City

What you're doing: Calculate total defaulted loan value per city in Crores. This is the number that converts "rate" into "cost." A 10% default rate on ₹50 Cr exposure is a ₹5 Cr problem. The same rate on ₹2 Cr exposure is a ₹20L problem. Not the same.

*Hint: SUM(loan_amount) WHERE status=Defaulted, grouped by city. Format as Crores. Also include total disbursed amount per city*

In [11]:
query = """
SELECT 
    dc.city_name AS `City`,
    CONCAT(ROUND(SUM(l.loan_amount) / 10000000.0, 2), ' Cr') AS `Disbursed Amounts`,
    CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN loan_amount * 1.0 ELSE 0 END) / 10000000.0, 2), ' Cr') AS `Defaulted Amounts`
FROM loans l
INNER JOIN customers c ON l.customer_id = c.customer_id
INNER JOIN dim_city dc ON c.city_id = dc.city_id
GROUP BY dc.city_name
ORDER BY ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN loan_amount * 1.0 ELSE 0 END) / 10000000.0, 2) DESC;
"""
start = time.perf_counter()
spark.sql(query).show(n=N)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+----------+-----------------+-----------------+
|      City|Disbursed Amounts|Defaulted Amounts|
+----------+-----------------+-----------------+
|    Ranchi|          6.71 Cr|           1.4 Cr|
|      Pune|          5.87 Cr|          1.14 Cr|
|   Lucknow|          7.51 Cr|          1.13 Cr|
|    Mumbai|          6.59 Cr|          1.06 Cr|
| Ahmedabad|          7.07 Cr|          1.05 Cr|
|    Indore|          6.73 Cr|          1.04 Cr|
|    Jaipur|          6.58 Cr|          1.03 Cr|
|     Kochi|          6.67 Cr|           1.0 Cr|
|     Jammu|          6.52 Cr|          0.99 Cr|
|    Rajkot|          6.32 Cr|          0.98 Cr|
|     Patna|          7.77 Cr|          0.84 Cr|
|    Bhopal|          8.53 Cr|          0.84 Cr|
|    Nashik|          5.62 Cr|          0.84 Cr|
|     Delhi|          8.22 Cr|          0.83 Cr|
|    Nagpur|          7.06 Cr|          0.82 Cr|
|    Raipur|           8.5 Cr|           0.8 Cr|
|    Shimla|          5.99 Cr|          0.76 Cr|
|   Gangtok|        

Does the city with the highest default RATE also have the highest defaulted AMOUNT? If not — this is the key insight of Phase 2.

- The city with highest default rate is 'Pune' which is 20.13%.
- The city with the highest default amount is 'Ranchi' is 1.4 Cr.

Query 2C: Financial Exposure by Knowledge Hub

- Calculate: Total portfolio value and total defaulted value per city.
- Business Purpose: Identify where the most capital is at risk.
- Formatting: Convert huge sums to Crores/Lakhs.

In [12]:
query = """
SELECT 
    dc.city_name AS `City`,
    COUNT(*) AS `Loans`,
    CONCAT(ROUND(SUM(l.loan_amount) / 10000000.0, 2), ' Cr') AS `Portfolio Values`, 
    CASE
        WHEN SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount * 1.0 ELSE 0 END) > 10000000.0 
        THEN CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount * 1.0 ELSE 0 END) / 10000000.0, 2), ' Cr')
        ELSE CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount * 1.0 ELSE 0 END) / 100000.0, 2), ' L')
    END AS `Defaulted Values`,
    CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount * 1.0 ELSE 0 END) * 100.0 / SUM(l.loan_amount), 2), '%') AS `Loss Rates (%)`
FROM loans l
INNER JOIN customers c ON l.customer_id = c.customer_id
INNER JOIN dim_city dc ON c.city_id = dc.city_id
GROUP BY dc.city_name
ORDER BY SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount * 1.0 ELSE 0 END) * 100.0 / SUM(l.loan_amount) DESC;
"""
start = time.perf_counter()
spark.sql(query).show(n=N)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+----------+-----+----------------+----------------+--------------+
|      City|Loans|Portfolio Values|Defaulted Values|Loss Rates (%)|
+----------+-----+----------------+----------------+--------------+
|    Ranchi|  168|         6.71 Cr|          1.4 Cr|        20.85%|
|      Pune|  149|         5.87 Cr|         1.14 Cr|        19.42%|
|    Mumbai|  168|         6.59 Cr|         1.06 Cr|        16.15%|
|    Jaipur|  160|         6.58 Cr|         1.03 Cr|        15.62%|
|    Rajkot|  166|         6.32 Cr|         97.93 L|        15.49%|
|    Indore|  170|         6.73 Cr|         1.04 Cr|        15.39%|
|     Jammu|  159|         6.52 Cr|         98.62 L|        15.12%|
|   Lucknow|  179|         7.51 Cr|         1.13 Cr|         15.1%|
|     Kochi|  165|         6.67 Cr|          1.0 Cr|        15.01%|
|    Nashik|  135|         5.62 Cr|         83.66 L|         14.9%|
| Ahmedabad|  160|         7.07 Cr|         1.05 Cr|        14.81%|
|    Shimla|  144|         5.99 Cr|         75.9

---

STEP 2D — Geographic Risk Pattern (State-Level Rollup)

What you're doing: Aggregate by state to find regional patterns. If multiple cities in the same state are defaulting — that suggests a macro factor (economy, employment, institutional quality) rather than a city-specific issue.

*Hint: GROUP BY state_name isntead of city_name. JOIN dim_city to get state. Show: total loans, defaults, default rate, total exposure per state*

In [13]:
query = """
SELECT 
    st.state_name AS `States`,
    st.region AS `Regions`,
    COUNT(*) AS `Loans`,
    COUNT(CASE WHEN l.loan_status = 'Defaulted' THEN 1 END) AS `Defaulted Loans`,
    CONCAT(ROUND(COUNT(CASE WHEN l.loan_status = 'Defaulted' THEN 1 END) * 100.0 / COUNT(*), 2), '%') AS `Default Rate (%)`,
    CASE
        WHEN SUM(CASE WHEN l.loan_status = 'Defaulted' THEN CAST(l.loan_amount AS DECIMAL) ELSE 0 END) >= 10000000.0
        THEN CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN CAST(l.loan_amount AS DECIMAL) ELSE 0 END) / 10000000.0, 2), ' Cr')
        ELSE CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN CAST(l.loan_amount AS DECIMAL) ELSE 0 END) / 100000.0, 2), ' L')
    END AS `Total Defaulted Exposure`,
    CASE
        WHEN SUM(CAST(l.loan_amount AS DECIMAL)) >= 10000000.0
        THEN CONCAT(ROUND(SUM(CAST(l.loan_amount AS DECIMAL)) / 10000000.0, 2), ' Cr')
        ELSE CONCAT(ROUND(SUM(CAST(l.loan_amount AS DECIMAL)) / 100000.0, 2), ' L')
    END AS `Total Portfolio Value`
FROM loans l
INNER JOIN customers c ON l.customer_id = c.customer_id
INNER JOIN dim_city dc ON c.city_id = dc.city_id
INNER JOIN dim_state st ON dc.state_id = st.state_id
GROUP BY st.state_name, st.region
ORDER BY SUM(CASE WHEN l.loan_status = 'Defaulted' THEN CAST(l.loan_amount AS DECIMAL) ELSE 0 END) DESC;
"""
start = time.perf_counter()
spark.sql(query).show(n=N)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+-----------------+---------+-----+---------------+----------------+------------------------+---------------------+
|           States|  Regions|Loans|Defaulted Loans|Default Rate (%)|Total Defaulted Exposure|Total Portfolio Value|
+-----------------+---------+-----+---------------+----------------+------------------------+---------------------+
|      Maharashtra|     West|  628|             93|          14.81%|                 3.86 Cr|             25.13 Cr|
|          Gujarat|     West|  479|             61|          12.73%|                 2.53 Cr|             20.04 Cr|
|   Madhya Pradesh|  Central|  374|             50|          13.37%|                 1.88 Cr|             15.26 Cr|
|    Uttar Pradesh|    North|  338|             40|          11.83%|                 1.68 Cr|             13.35 Cr|
|        Jharkhand|     East|  168|             30|          17.86%|                 1.40 Cr|              6.71 Cr|
|        Rajasthan|    North|  160|             26|          16.25%|    

Which state has the highest total defaulted exposure? What business question does this answer that city-level data cannot?

> Maharashtra with ₹3.86 Crores has the highest total defaulted exposure which also answers the question "Is this a regional crisis?" with a YES. Multiple cities across Maharastra are defaulting at similar rates indicating the fall of an entire state, making the problem MACRO, not MICRO.

Query 2D: Geographic Risk Classification

- Calculate: Classify cities as 'Critical', 'High Risk', 'Monitor', or 'Safe' based on default rates.
- Business Purpose: Prioritize regional intervention teams.
- Rules: defaults > 15% = Critical, > 10% = High Risk.

In [14]:
query = """
SELECT 
    dc.city_name AS `City`,
    COUNT(*) AS `Loans`,
    CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100 / COUNT(*), 2), '%') AS `Default Rates (%)`,
    CASE
        WHEN ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100 / COUNT(*), 2) > 15 THEN 'Critical'
        WHEN ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100 / COUNT(*), 2) > 10 THEN 'High Risk'
        WHEN ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100 / COUNT(*), 2) > 5 THEN 'Monitor'
        ELSE 'Safe'
    END AS `Risk Category`
FROM loans l
INNER JOIN customers c ON l.customer_id = c.customer_id
INNER JOIN dim_city dc ON c.city_id = dc.city_id
GROUP BY dc.city_name;
"""
start = time.perf_counter()
spark.sql(query).show(n=N)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+-----------+-----+-----------------+-------------+
|       City|Loans|Default Rates (%)|Risk Category|
+-----------+-----+-----------------+-------------+
|  Bangalore|  148|            8.78%|      Monitor|
|      Kochi|  165|           15.15%|     Critical|
|Bhubaneswar|  199|            9.55%|      Monitor|
|      Jammu|  159|           14.47%|    High Risk|
|      Patna|  183|            9.29%|      Monitor|
|   Vadodara|  153|            7.19%|      Monitor|
|    Gangtok|  150|           11.33%|    High Risk|
|    Chennai|  191|            6.81%|      Monitor|
|    Lucknow|  179|           13.97%|    High Risk|
|     Shimla|  144|           13.19%|    High Risk|
|     Ranchi|  168|           17.86%|     Critical|
|     Mumbai|  168|            13.1%|    High Risk|
|  Ahmedabad|  160|            15.0%|    High Risk|
|    Kolkata|  129|            8.53%|      Monitor|
|   Dehradun|  163|            9.82%|      Monitor|
|       Pune|  149|           20.13%|     Critical|
|      Delhi

---

STEP 2E — Regional Crisis Dashboard (Final Query)

What you're doing: Combine city volume, default rate, and financial exposure. Filter to cities with 100+ loans (statistical significance). Sort by financial impact — not default rate. This is your final city-level recommendation output.

*Hint: Full dashboard: city_name, total_loans, default_rate, defaulted_exposure_cr, exposure_rank. HAVING COUNT >= 100. ORDER BY financial exposure DESC. LIMIT top 15* 

In [15]:
query = """
SELECT 
    City,
    State,
    Region,
    `Loans`,
    `Defaulted Loans`,
    `Default Rates (%)`,
    `Defaulted Values`,
    `Risk Category`,
    `Exposure Rank`
FROM (
    SELECT 
        dc.city_name AS `City`,
        st.state_name AS `State`,
        st.region AS `Region`,
        COUNT(*) AS `Loans`,
        COUNT(CASE WHEN l.loan_status = 'Defaulted' THEN 1 END) AS `Defaulted Loans`,
        CONCAT(ROUND(COUNT(CASE WHEN l.loan_status = 'Defaulted' THEN 1 END) * 100.0 / COUNT(*), 2), '%') AS `Default Rates (%)`,
        CASE
            WHEN SUM(CASE WHEN l.loan_status = 'Defaulted' THEN CAST(l.loan_amount AS DECIMAL) ELSE 0 END) >= 10000000.0
            THEN CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN CAST(l.loan_amount AS DECIMAL) ELSE 0 END) / 10000000.0, 2), ' Cr')
            ELSE CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN CAST(l.loan_amount AS DECIMAL) ELSE 0 END) / 100000.0, 2), ' L')
        END AS `Defaulted Values`,
        CASE
            WHEN COUNT(CASE WHEN l.loan_status = 'Defaulted' THEN 1 END) * 100.0 / COUNT(*) >= 15.0
            AND SUM(CASE WHEN l.loan_status = 'Defaulted' THEN CAST(l.loan_amount AS DECIMAL) ELSE 0 END) >= 100000000.0
            THEN 'CRITICAL - Halt Immediately'
            WHEN COUNT(CASE WHEN l.loan_status = 'Defaulted' THEN 1 END) * 100.0 / COUNT(*) >= 14.0
            OR SUM(CASE WHEN l.loan_status = 'Defaulted' THEN CAST(l.loan_amount AS DECIMAL) ELSE 0 END) >= 80000000.0
            THEN 'HIGH RISK - Review Urgently'
            WHEN COUNT(CASE WHEN l.loan_status = 'Defaulted' THEN 1 END) * 100.0 / COUNT(*) >= 12.0
            OR SUM(CASE WHEN l.loan_status = 'Defaulted' THEN CAST(l.loan_amount AS DECIMAL) ELSE 0 END) >= 50000000.0
            THEN 'MODERATE - Monitor Closely'
            ELSE 'LOW - Continue Operations'
        END AS `Risk Category`,
        ROW_NUMBER() OVER (ORDER BY SUM(CASE WHEN l.loan_status = 'Defaulted' THEN CAST(l.loan_amount AS DECIMAL) ELSE 0 END) DESC) AS `Exposure Rank`
    FROM loans l
    INNER JOIN customers c ON l.customer_id = c.customer_id
    INNER JOIN dim_city dc ON c.city_id = dc.city_id
    INNER JOIN dim_state st ON dc.state_id = st.state_id
    GROUP BY dc.city_name, st.state_name, st.region
    HAVING COUNT(*) >= 100
)
ORDER BY `Exposure Rank`
LIMIT 15
"""
start = time.perf_counter()
spark.sql(query).show(n=N)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+---------+-----------------+-------+-----+---------------+-----------------+----------------+--------------------+-------------+
|     City|            State| Region|Loans|Defaulted Loans|Default Rates (%)|Defaulted Values|       Risk Category|Exposure Rank|
+---------+-----------------+-------+-----+---------------+-----------------+----------------+--------------------+-------------+
|   Ranchi|        Jharkhand|   East|  168|             30|           17.86%|         1.40 Cr|HIGH RISK - Revie...|            1|
|     Pune|      Maharashtra|   West|  149|             30|           20.13%|         1.14 Cr|HIGH RISK - Revie...|            2|
|  Lucknow|    Uttar Pradesh|  North|  179|             25|           13.97%|         1.13 Cr|MODERATE - Monito...|            3|
|   Mumbai|      Maharashtra|   West|  168|             22|           13.10%|         1.06 Cr|MODERATE - Monito...|            4|
|Ahmedabad|          Gujarat|   West|  160|             24|           15.00%|         1.05

Name your top 3 halt cities. Now check: if you had sorted by default rate instead of exposure — would you have recommended the same 3?

> The top 3 halt cities are:

- Ranchi
- Pune
- Lucknow

Query 2E: Regional Crisis Dashboard

- Calculate: Comprehensive city-level dashboard combining volume, risk, and financial impact.
- Business Purpose: Strategic decision support for regional shutdowns.
- Sorting: Sort by "Total Losses" (Defaulted Value) to prioritize highest financial damage.
- REQUIRED Text Summary: Recommend 3 cities to halt lending in immediately.

In [16]:
query = """
WITH city_metrics AS (
    SELECT 
        dc.city_name AS City,
        COUNT(*) AS `Loans`,
        COUNT(DISTINCT c.customer_id) AS `Customers`,
        SUM(l.loan_amount) AS total_portfolio,
        SUM(CASE 
            WHEN l.loan_status = 'Defaulted' 
            THEN l.loan_amount * 1.0 
            ELSE 0 
        END) AS default_exposure,
        ROUND(SUM(CASE 
            WHEN l.loan_status = 'Defaulted' THEN 1 
            ELSE 0 
        END) * 100.0 / COUNT(*), 2) AS default_rate
    FROM loans l
    INNER JOIN customers c ON l.customer_id = c.customer_id
    INNER JOIN dim_city dc ON c.city_id = dc.city_id
    GROUP BY dc.city_name
    HAVING COUNT(*) >= 100
)

SELECT 
    City,
    Loans,
    Customers,
    CONCAT(ROUND(total_portfolio / 10000000.0, 2), ' Cr') AS `Portfolio Value`,
    CASE
        WHEN default_exposure > 10000000 
        THEN CONCAT(ROUND(default_exposure / 10000000.0, 2), ' Cr')
        ELSE CONCAT(ROUND(default_exposure / 100000.0, 2), ' L')
    END AS `Defaulted Value`,
    CONCAT(ROUND(default_rate, 2), '%') AS `Default Rates (%)`,
    ROUND((default_exposure / 10000000.0) * default_rate, 2) AS `Exposure Score`,
    RANK() OVER (
        ORDER BY (default_exposure / 10000000.0) * default_rate DESC
    ) AS `Exposure Rank`,
    CASE NTILE(4) OVER (ORDER BY default_exposure DESC)
        WHEN 1 THEN 'Critical'
        WHEN 2 THEN 'High'
        WHEN 3 THEN 'Moderate'
        ELSE 'Low'
    END AS `Risk Quartile`
FROM city_metrics
ORDER BY default_exposure DESC
LIMIT 15;
"""
start = time.perf_counter()
spark.sql(query).show()
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+---------+-----+---------+---------------+---------------+-----------------+--------------+-------------+-------------+
|     City|Loans|Customers|Portfolio Value|Defaulted Value|Default Rates (%)|Exposure Score|Exposure Rank|Risk Quartile|
+---------+-----+---------+---------------+---------------+-----------------+--------------+-------------+-------------+
|   Ranchi|  168|      103|        6.71 Cr|         1.4 Cr|           17.86%|         24.97|            1|     Critical|
|     Pune|  149|       94|        5.87 Cr|        1.14 Cr|           20.13%|         22.95|            2|     Critical|
|  Lucknow|  179|      101|        7.51 Cr|        1.13 Cr|           13.97%|         15.85|            5|     Critical|
|   Mumbai|  168|      106|        6.59 Cr|        1.06 Cr|           13.10%|         13.94|           10|     Critical|
|Ahmedabad|  160|       97|        7.07 Cr|        1.05 Cr|           15.00%|         15.71|            6|     Critical|
|   Indore|  170|      104|     

> The 3 cities to halt lending it immediately are Ranchi, Pune and Lucknow due to their high defaulted value exposure.

---

### BUILD YOUR CUSTOM COLUMNS — Required for Tuesday Submission

> NEW TECHNIQUE — Window Functions: Phase 1 used CASE for classification. Today you unlock RANK(), DENSE_RANK(), and NTILE() — SQL's ranking engine. These functions compute a value ACROSS rows without collapsing them. AI can write a CASE. Only an analyst can choose the right ranking logic.

Column 1 — exposure_rank: Use RANK() OVER (ORDER BY ...) to rank cities by the PRODUCT of default_rate × total_defaulted_exposure. Not rate alone. Not exposure alone. The combined risk signal is the action signal. You must use a window function — not CASE.

Column 2 — risk_quartile: Use NTILE(4) OVER (ORDER BY ...) to divide cities into 4 risk quartiles based on defaulted exposure. Label them: Q1=Critical, Q2=High, Q3=Moderate, Q4=Low. Why NTILE and not CASE? Because NTILE adapts automatically — if 5 new cities enter the portfolio next month, the quartile boundaries shift. Your CASE thresholds from Monday would break.

Your exposure_rank using RANK() OVER (...):

*Hint: RANK() OVER (ORDER BY (default_rate * default_exposure) DESC) AS exposure_rank. Why RANK and not ROW_NUMBER? What happens when cities tie?* 

In [17]:
query = """
WITH city_metrics AS (
    SELECT 
        dc.city_name AS City,
        COUNT(*) AS `Loans`,
        COUNT(DISTINCT c.customer_id) AS `Customers`,
        SUM(l.loan_amount) AS total_portfolio,
        SUM(CASE 
            WHEN l.loan_status = 'Defaulted' 
            THEN l.loan_amount * 1.0 
            ELSE 0 
        END) AS default_exposure,
        SUM(CASE 
            WHEN l.loan_status = 'Defaulted' THEN 1 
            ELSE 0 
        END) * 100.0 / COUNT(*) AS default_rate
    FROM loans l
    INNER JOIN customers c ON l.customer_id = c.customer_id
    INNER JOIN dim_city dc ON c.city_id = dc.city_id
    GROUP BY dc.city_name
    HAVING COUNT(*) >= 100
)

SELECT 
    City,
    Loans,
    Customers,
    CONCAT(ROUND(total_portfolio / 10000000.0, 2), ' Cr') AS `Portfolio Value`,
    CASE
        WHEN default_exposure > 10000000 
        THEN CONCAT(ROUND(default_exposure / 10000000.0, 2), ' Cr')
        ELSE CONCAT(ROUND(default_exposure / 100000.0, 2), ' L')
    END AS `Defaulted Value`,
    CONCAT(ROUND(default_rate, 2), '%') AS `Default Rates (%)`,
    RANK() OVER (
        ORDER BY default_exposure * default_rate DESC
    ) AS `Exposure Rank`
FROM city_metrics
ORDER BY default_exposure DESC;
"""
start = time.perf_counter()
spark.sql(query).show(n=N)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+----------+-----+---------+---------------+---------------+-----------------+-------------+
|      City|Loans|Customers|Portfolio Value|Defaulted Value|Default Rates (%)|Exposure Rank|
+----------+-----+---------+---------------+---------------+-----------------+-------------+
|    Ranchi|  168|      103|        6.71 Cr|         1.4 Cr|           17.86%|            1|
|      Pune|  149|       94|        5.87 Cr|        1.14 Cr|           20.13%|            2|
|   Lucknow|  179|      101|        7.51 Cr|        1.13 Cr|           13.97%|            5|
|    Mumbai|  168|      106|        6.59 Cr|        1.06 Cr|           13.10%|           10|
| Ahmedabad|  160|       97|        7.07 Cr|        1.05 Cr|           15.00%|            6|
|    Indore|  170|      104|        6.73 Cr|        1.04 Cr|           16.47%|            3|
|    Jaipur|  160|       90|        6.58 Cr|        1.03 Cr|           16.25%|            4|
|     Kochi|  165|      105|        6.67 Cr|         1.0 Cr|          

Your risk_quartile using NTILE(4) OVER (...):

*Hint: NTILE(4) OVER (ORDER BY defaulted_exposure DESC) AS risk_quartile. Then: CASE risk_quartile WHEN 1 THEN 'Critical' WHEN 2 THEN 'HIGH' WHEN 3 THEN 'MODERATE' ELSE 'LOW' END*


In [18]:
query = """
WITH city_metrics AS (
    SELECT 
        dc.city_name AS City,
        COUNT(*) AS `Loans`,
        COUNT(DISTINCT c.customer_id) AS `Customers`,
        SUM(l.loan_amount) AS total_portfolio,
        SUM(CASE 
            WHEN l.loan_status = 'Defaulted' 
            THEN l.loan_amount * 1.0 
            ELSE 0 
        END) AS default_exposure,
        ROUND(SUM(CASE 
            WHEN l.loan_status = 'Defaulted' THEN 1 
            ELSE 0 
        END) * 100.0 / COUNT(*), 2) AS default_rate
    FROM loans l
    INNER JOIN customers c ON l.customer_id = c.customer_id
    INNER JOIN dim_city dc ON c.city_id = dc.city_id
    GROUP BY dc.city_name
    HAVING COUNT(*) >= 100
)

SELECT 
    City,
    Loans,
    Customers,
    CONCAT(ROUND(total_portfolio / 10000000.0, 2), ' Cr') AS `Portfolio Value`,
    CASE
        WHEN default_exposure > 10000000 
        THEN CONCAT(ROUND(default_exposure / 10000000.0, 2), ' Cr')
        ELSE CONCAT(ROUND(default_exposure / 100000.0, 2), ' L')
    END AS `Defaulted Value`,
    CONCAT(ROUND(default_rate, 2), '%') AS `Default Rates (%)`,
    ROUND((default_exposure / 10000000.0) * default_rate, 2) AS `Exposure Score`,
    RANK() OVER (
        ORDER BY (default_exposure / 10000000.0) * (total_portfolio / 10000000.0) * default_rate DESC
    ) AS `Exposure Rank`,
    CASE NTILE(4) OVER (ORDER BY default_exposure DESC)
        WHEN 1 THEN 'Critical'
        WHEN 2 THEN 'High'
        WHEN 3 THEN 'Moderate'
        ELSE 'Low'
    END AS `Risk Quartile`
FROM city_metrics
ORDER BY (default_exposure / 10000000.0) * (total_portfolio / 10000000.0) * default_rate DESC;
"""
start = time.perf_counter()
spark.sql(query).show(n=N)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+-----------+-----+---------+---------------+---------------+-----------------+--------------+-------------+-------------+
|       City|Loans|Customers|Portfolio Value|Defaulted Value|Default Rates (%)|Exposure Score|Exposure Rank|Risk Quartile|
+-----------+-----+---------+---------------+---------------+-----------------+--------------+-------------+-------------+
|     Ranchi|  168|      103|        6.71 Cr|         1.4 Cr|           17.86%|         24.97|            1|     Critical|
|       Pune|  149|       94|        5.87 Cr|        1.14 Cr|           20.13%|         22.95|            2|     Critical|
|    Lucknow|  179|      101|        7.51 Cr|        1.13 Cr|           13.97%|         15.85|            3|     Critical|
|     Indore|  170|      104|        6.73 Cr|        1.04 Cr|           16.47%|         17.07|            4|     Critical|
|  Ahmedabad|  160|       97|        7.07 Cr|        1.05 Cr|           15.00%|         15.71|            5|     Critical|
|     Jaipur|  1

Why is NTILE better than hardcoded CASE thresholds here? When would CASE be better? (2–3 sentences):

*Hint: Compare: NTILE adapts to data changes automatically. Your phase 1 CASE used fixed thresholds. When is each approach stronger?*

> NTILE is better than hardcoded CASE thresholds because it automatically splits the cities into equal sized groups (quartiles) and also automatically adapts when new cities are added. Also since our portfolio data can grow or shrink in the near future, the hardcoded thresholds may become outdated making the user update the query time and again. However, in cases where you must stay consistent due to company policies, it is mandatory to use CASE thresholds as they can strictly follow absolute business rules.

---

### BEFORE YOU SUBMIT — Answer These Three Questions

1. If you sorted your final list by default rate instead of exposure_rank — does your top-3 halt recommendation change? Name the specific cities that move in or out, and explain what that difference means for the CEO's decision.

> The top 3 halt recommendation changes from ('Ranchi', 'Pune', 'Lucknow') using total losses (defaulted amount) to ('Pune', 'Ranchi', 'Indore') using default_rank. Lucknow exits the halt list despite being the third-largest source of losses (1.13 Cr), while Indore enters despite lower absolute exposure (1.04 Cr). This means the CEO would halt cities based on frequency (rate) rather than financial damage (exposure), continuing to lend in a city that has already accumulated 1.13 Crores in losses.  For a halt decision, total loss rank is correct because the CEO asked "where are we bleeding money" — not "which cities have high rates."

2. You excluded cities with fewer than 100 loans. Name one city that was excluded and explain: why does excluding it protect the business from making a bad decision?

> No, city was excluded as there were no cities with fewer than 100 loans.

3. Write your 3-city halt decision memo. Format: City Name → default rate → defaulted exposure → specific reason to halt THIS city and not the next one on the list.

- 'Ranchi' → 17.86% → 1.4 Cr → It is the highest financial exposure in the portfolio (1.40 Cr), combined with one of the highest default rates.
- 'Pune' → 20.13% → 1.14 Cr → It has the highest default rate (20.13%) from our entire list, while also having a decent financial exposure, making it the city with the worst failure frequency.
- 'Lucknow' → 13.97% → 1.13 Cr → It has the 3rd highest financial exposure of 1.13 Cr

---

### Geographic Crisis Analysis — CEO Response

> Where We Are Bleeding Money ?

We are bleeding money primarily in the West region, where Maharashtra and Gujarat combined account for ₹6.39 Crores in defaults—18% of total portfolio loss. Within Maharashtra, the crisis is distributed across Mumbai (₹1.06 Cr), Pune (₹1.14 Cr), Nashik (₹83.66 L), and Nagpur (₹82.25 L), indicating this is not a single-city problem but a state-wide market failure. Gujarat shows similar patterns with Ahmedabad (₹1.05 Cr) and Rajkot (₹97.93 L) both experiencing elevated defaults.

Beyond the West, three other cities stand out as critical individual sources of loss: Ranchi in Jharkhand (₹1.40 Cr—the single largest loss), Lucknow in Uttar Pradesh (₹1.13 Cr despite a lower default rate), and Indore in Madhya Pradesh (₹1.04 Cr). Together, these seven cities (Ranchi, Pune, Lucknow, Indore, Ahmedabad, Rajkot, and Mumbai) account for ₹7.82 Crores in defaults—22% of all portfolio losses. The remaining 8 cities in our top-15 list add another ₹2.36 Crores, bringing the geographic concentration to 32% of losses in just 15 cities out of our entire portfolio.

---

> Which Cities Have the Worst Default Rates ?

Pune has the worst default rate in the entire portfolio at 20.13%, meaning one in every five borrowers is defaulting. This is followed by Ranchi (17.86%), Indore (16.47%), Jaipur (16.25%), Rajkot (15.66%), Kochi (15.15%), Ahmedabad (15.00%), and Jammu (14.47%). All eight of these cities exceed our portfolio average of 11.80% and represent systematic quality failures, not random market noise.

However, the CEO's question "which cities have the worst rates" cannot be answered in isolation. Pune's 20.13% rate on 149 loans is severe, but it generates ₹1.14 Crores in loss. Ranchi's 17.86% rate on 168 loans generates ₹1.40 Crores in loss—higher absolute damage despite a lower rate. Lucknow presents an even more important case: with only a 13.97% default rate (below many cities in this list), it still contributes ₹1.13 Crores in loss because of its large portfolio size (179 loans, ₹7.51 Cr deployed). The rate alone is misleading; the financial impact (rate multiplied by exposure) is what drives the halt decision.

---

> Which Geographic Markets Are Causing This Crisis ?

The West region is causing the crisis. Maharashtra (₹3.86 Cr loss across four cities) and Gujarat (₹2.53 Cr loss across two cities) together represent 29% of portfolio losses. The uniformity of high default rates across multiple cities in both states—Pune (20.13%), Mumbai (13.10%), Nashik (14.07%), Nagpur (12.50%) in Maharashtra, and Ahmedabad (15%), Rajkot (15.66%) in Gujarat—suggests this is not a problem of individual city underwriting failures but a regional macro-economic factor. Possible causes include recession in these states, collapse of the education market, reduced employer demand for graduates, or credit market saturation.

The North region is emerging as a secondary concern, with Jaipur (16.25%), Lucknow (13.97%), Rajkot (15.66%), and Jammu (14.47%) all showing elevated defaults across multiple states. The Central region shows Indore (16.47%) as a critical outlier. The South region, which was expected to be safer, shows weakness in Kochi (15.15%) that contradicts the earlier state-level data showing Kerala and Tamil Nadu with lower rates—suggesting institution-specific or city-specific issues rather than state-wide problems.

> Regional Strategy Adjustment Required ?

To adjust regional strategies, the board should immediately halt lending in the West region's top three cities (Ranchi, Pune, Lucknow) and implement enhanced collections and recovery programs. Within 30 days, halt lending should expand to Indore, Jaipur, Ahmedabad, and Kochi unless root-cause analysis reveals correctable origination or institution-partnership issues. For cities with lower default rates but large portfolios (Delhi at 10.53%, Bhopal at 10.78%, Patna at 9.29%), continue standard monitoring but tighten underwriting and implement weekly risk reviews.

At the state level, investigate what changed in Maharashtra and Gujarat in the past 12-18 months. Did major employers reduce hiring? Did colleges close or lose accreditation? Did credit markets freeze? If the crisis is macro-economic, no city-level halt will solve it—the board must exit these states entirely. If the crisis is institution-specific (certain colleges are producing unemployable graduates), the board should renegotiate terms with those institutions or exit partnerships. If the crisis is origination-specific (underwriting standards deteriorated), the board should rebuild underwriting teams and pilot new borrower-selection criteria in a small subset of cities before national rollout.

The data shows that Pune, Ranchi, and Lucknow are the immediate halt priorities. But the board's strategic decision—whether to exit regions, renegotiate with institutions, or rebuild underwriting—depends on understanding why the West region and North region are failing. That analysis is the next phase of the crisis response.